# LAB 2: TRỰC QUAN HÓA DỮ LIỆU - XỬ LÝ ĐIỂM THI ĐẠI HỌC

**Sinh viên**: Trần Tấn Phát - MSSV: 2274802010644

## Ý Nghĩa của Lab 2

Lab 2 tập trung vào kỹ năng xử lý, thống kê và trực quan hóa dữ liệu thực tế (điểm thi đại học). Thông qua bài tập này, bạn sẽ nắm vững:

- **Thống kê mô tả**: Biết cách sắp xếp, tính toán các chỉ số đặc trưng (trung bình, trung vị, phân vị...) bằng Pivot Table.
- **Trực quan hóa cơ bản**: Sử dụng biểu đồ cột, tròn để mô tả tần số/tần suất của các biến định tính như Giới tính (GT), Khu vực (KV).
- **Phân tích nhóm**: So sánh dữ liệu giữa các nhóm đối tượng (ví dụ: xếp loại học lực của học sinh nữ).
- **Phân tích phân phối và tương quan**: Xác định hình dáng phân phối của điểm số qua biểu đồ Box-plot, Histogram và khảo sát mối liên hệ giữa các môn học.

## Import Required Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis, shapiro
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Load the data
file_path = '/Users/trantanphat/Documents/Python/PTDL_DL/TH/thư mục không có tiêu đề/processed_dulieuxettuyendaihoc.csv'
df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nBasic statistics:")
print(df.describe())

: 

# PHẦN 1: THỐNG KÊ DỮ LIỆU

## 1.1 Sắp xếp dữ liệu DH1 theo thứ tự tăng dần

In [ ]:
# 1.1 Sắp xếp DH1 tăng dần
sorted_dh1 = df.sort_values('DH1', ascending=True)[['STT', 'DH1']]
print("DH1 sorted ascending:")
print(sorted_dh1)
print(f"\nMin DH1: {df['DH1'].min()}, Max DH1: {df['DH1'].max()}")

: 

## 1.2 Sắp xếp DH2 tăng dần theo nhóm giới tính

In [ ]:
# 1.2 Sắp xếp DH2 tăng dần theo giới tính
sorted_dh2_gt = df.sort_values(['GT', 'DH2'], ascending=[True, True])[['STT', 'GT', 'DH2']]
print("DH2 sorted by Gender and ascending:")
print(sorted_dh2_gt)

# Count by gender
print("\nCount by Gender:")
print(df['GT'].value_counts())

## 1.3 Pivot Table thống kê DH1 theo KT

In [ ]:
# 1.3 Pivot Table DH1 theo KT
pivot_dh1_kt = pd.pivot_table(
    df, 
    values='DH1', 
    index='KT',
    aggfunc=['count', 'sum', 'mean', 'median', 'min', 'max', 'std',
             ('Q1', lambda x: x.quantile(0.25)),
             ('Q2', lambda x: x.quantile(0.50)),
             ('Q3', lambda x: x.quantile(0.75))]
)
print("Statistics of DH1 by KT (Exam Type):")
print(pivot_dh1_kt.round(4))

## 1.4 Pivot Table thống kê DH1 theo KT và KV

In [ ]:
# 1.4 Pivot Table DH1 theo KT và KV
def get_stats(x):
    return pd.Series({
        'count': x.count(),
        'sum': x.sum(),
        'mean': x.mean(),
        'median': x.median(),
        'min': x.min(),
        'max': x.max(),
        'std': x.std(),
        'Q1': x.quantile(0.25),
        'Q2': x.quantile(0.50),
        'Q3': x.quantile(0.75)
    })

pivot_dh1_kt_kv = df.groupby(['KT', 'KV'])['DH1'].apply(get_stats)
print("Statistics of DH1 by KT and KV:")
print(pivot_dh1_kt_kv.round(4))

## 1.5 Pivot Table thống kê DH1 theo KT, KV và DT

In [ ]:
# 1.5 Pivot Table DH1 theo KT, KV và DT
pivot_dh1_kt_kv_dt = df.groupby(['KT', 'KV', 'DT'])['DH1'].apply(get_stats)
print("Statistics of DH1 by KT, KV, and DT:")
print(pivot_dh1_kt_kv_dt.round(4))

# PHẦN 2: TRÌNH BÀY DỮ LIỆU

## 2.1 Trình bày dữ liệu biến GT (Giới tính)

In [ ]:
# 2.1 Frequency table for GT (Gender)
gt_freq = df['GT'].value_counts()
gt_relative = df['GT'].value_counts(normalize=True) * 100

freq_table_gt = pd.DataFrame({
    'Frequency': gt_freq,
    'Relative Frequency (%)': gt_relative.round(2)
})
print("Frequency and Relative Frequency Table for GT (Gender):")
print(freq_table_gt)

# Visualizations for GT
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
gt_freq.plot(kind='bar', ax=axes[0], color=['#FF6B9D', '#4A90E2'])
axes[0].set_title('Frequency Distribution of Gender', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
colors = ['#FF6B9D', '#4A90E2']
axes[1].pie(gt_freq.values, labels=gt_freq.index, autopct='%1.1f%%', 
            colors=colors, startangle=90)
axes[1].set_title('Relative Frequency Distribution of Gender', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 2.2 Trình bày dữ liệu biến US_TBM1, US_TBM2, US_TBM3

In [ ]:
# 2.2 Statistics and visualization for US_TBM1, US_TBM2, US_TBM3
us_vars = ['US_TBM1', 'US_TBM2', 'US_TBM3']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, var in enumerate(us_vars):
    freq = df[var].value_counts().sort_index()
    axes[i].bar(freq.index, freq.values, color='skyblue', edgecolor='navy')
    axes[i].set_title(f'Frequency Distribution of {var}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Count')
    axes[i].grid(axis='y', alpha=0.3)
    
    # Print frequency table
    print(f"\n{var} Frequency Table:")
    print(freq)
    print(f"Mean: {df[var].mean():.4f}, Std: {df[var].std():.4f}")

plt.tight_layout()
plt.show()

## 2.3 Trình bày DT cho học sinh nam

In [ ]:
# 2.3 DT for male students
male_dt = df[df['GT'] == 'M']['DT']
print("DT (Ethnicity) for Male Students:")
print(male_dt.value_counts())

fig, ax = plt.subplots(figsize=(10, 6))
male_dt.value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='navy')
ax.set_title('Ethnicity Distribution of Male Students', fontsize=12, fontweight='bold')
ax.set_xlabel('Ethnicity')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 2.4 Trình bày KV cho nam, dân tộc Kinh, điểm thỏa điều kiện

In [ ]:
# 2.4 KV for male, Kinh ethnicity, with score conditions
filtered_kv = df[(df['GT'] == 'M') & (df['DT'] == 0.0) & 
                  (df['DH1'] >= 5.0) & (df['DH2'] >= 4.0) & (df['DH3'] >= 4.0)]['KV']

print("KV (Region) for Male, Kinh Ethnicity, Meeting Score Conditions:")
print(filtered_kv.value_counts())
print(f"\nTotal records meeting criteria: {len(filtered_kv)}")

if len(filtered_kv) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    filtered_kv.value_counts().plot(kind='bar', ax=ax, color='coral', edgecolor='darkred')
    ax.set_title('Region (KV) Distribution - Male, Kinh, Meeting Conditions', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Region')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=0)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

## 2.5 Trình bày DH1, DH2, DH3 >= 5.0 thuộc khu vực 2NT

In [ ]:
# 2.5 DH1, DH2, DH3 >= 5.0 in region 2NT
filtered_2nt = df[(df['KV'] == '2NT') & (df['DH1'] >= 5.0) & (df['DH2'] >= 5.0) & (df['DH3'] >= 5.0)]
print("Students with scores >= 5.0 in region 2NT:")
print(f"Total: {len(filtered_2nt)} records")
print("\nStatistics:")
print(filtered_2nt[['DH1', 'DH2', 'DH3']].describe())

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, var in enumerate(['DH1', 'DH2', 'DH3']):
    axes[i].hist(filtered_2nt[var], bins=10, color='lightgreen', edgecolor='darkgreen')
    axes[i].set_title(f'Distribution of {var} (Region 2NT, >= 5.0)', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Frequency')
    axes[i].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# PHẦN 3: TRỰC QUAN HÓA DỮ LIỆU THEO NHÓM PHÂN LOẠI

## 3.1 Học sinh nữ trên nhóm XL1, XL2, XL3 (unstacked)

In [ ]:
# 3.1 Female students by classification (XL1, XL2, XL3)
female_data = df[df['GT'] == 'F']

# Create pivot table for counting
xl_counts = pd.crosstab(
    [female_data['XL1'], female_data['XL2'], female_data['XL3']], 
    columns='Count'
).reset_index()

# Better approach: count by each classification level
xl_levels = ['XL1', 'XL2', 'XL3']
colors_map = {'Y': '#FFD700', 'TB': '#87CEEB', 'K': '#90EE90', 'G': '#FF6B6B', 'XS': '#FFA500'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, xl in enumerate(xl_levels):
    counts = female_data[xl].value_counts()
    colors = [colors_map.get(level, '#CCCCCC') for level in counts.index]
    
    axes[idx].bar(counts.index, counts.values, color=colors, edgecolor='black', linewidth=1.5)
    axes[idx].set_title(f'Female Students - {xl} Classification', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(f'{xl} Level')
    axes[idx].set_ylabel('Number of Students')
    axes[idx].grid(axis='y', alpha=0.3)
    
    print(f"\n{xl} distribution for female students:")
    print(counts)

plt.tight_layout()
plt.show()

## 3.2 KQXT cho khối A, A1, B trong khu vực 1, 2

In [ ]:
# 3.2 KQXT for exam types A, A1, B in regions 1, 2
filtered_aab_kv12 = df[
    (df['KT'].isin(['A', 'A1', 'B'])) & 
    (df['KV'].isin([1, 2, '1', '2']))
]

kqxt_counts = pd.crosstab(filtered_aab_kv12['KT'], filtered_aab_kv12['KQXT'])
print("KQXT distribution for exam types A, A1, B in regions 1, 2:")
print(kqxt_counts)

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
kqxt_counts.plot(kind='bar', ax=ax, color=['#FF6B6B', '#90EE90'], width=0.7)
ax.set_title('Admission Results (KQXT) by Exam Type (A, A1, B)', fontsize=12, fontweight='bold')
ax.set_xlabel('Exam Type')
ax.set_ylabel('Number of Candidates')
ax.legend(['Failed', 'Passed'], loc='upper right')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3.3 Số lượng thí sinh theo khu vực và khối thi

In [ ]:
# 3.3 Number of candidates by region and exam type
candidates_by_kv_kt = pd.crosstab(df['KV'], df['KT'])
print("Number of candidates by Region (KV) and Exam Type (KT):")
print(candidates_by_kv_kt)

fig, ax = plt.subplots(figsize=(12, 6))
candidates_by_kv_kt.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Number of Candidates by Region and Exam Type', fontsize=12, fontweight='bold')
ax.set_xlabel('Region')
ax.set_ylabel('Number of Candidates')
ax.legend(title='Exam Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3.4 Số thí sinh đậu, rớt theo khối thi

In [ ]:
# 3.4 Passed/Failed by exam type
# Map KQXT: 0.0=Failed, 1.0=Passed
kqxt_by_kt = pd.crosstab(df['KT'], df['KQXT'])
print("Pass/Fail by Exam Type:")
print(kqxt_by_kt)

fig, ax = plt.subplots(figsize=(12, 6))
kqxt_by_kt.plot(kind='bar', ax=ax, color=['#FF6B6B', '#90EE90'], width=0.7)
ax.set_title('Number of Passed and Failed Candidates by Exam Type', fontsize=12, fontweight='bold')
ax.set_xlabel('Exam Type')
ax.set_ylabel('Number of Candidates')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(['Failed', 'Passed'], loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3.5 Số thí sinh đậu, rớt theo khu vực

In [ ]:
# 3.5 Passed/Failed by region
kqxt_by_kv = pd.crosstab(df['KV'], df['KQXT'])
print("Pass/Fail by Region:")
print(kqxt_by_kv)

fig, ax = plt.subplots(figsize=(12, 6))
kqxt_by_kv.plot(kind='bar', ax=ax, color=['#FF6B6B', '#90EE90'], width=0.7)
ax.set_title('Number of Passed and Failed Candidates by Region', fontsize=12, fontweight='bold')
ax.set_xlabel('Region')
ax.set_ylabel('Number of Candidates')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
ax.legend(['Failed', 'Passed'], loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3.6 Số thí sinh đậu, rớt theo dân tộc

In [ ]:
# 3.6 Passed/Failed by ethnicity
kqxt_by_dt = pd.crosstab(df['DT'], df['KQXT'])
print("Pass/Fail by Ethnicity:")
print(kqxt_by_dt)

fig, ax = plt.subplots(figsize=(12, 6))
kqxt_by_dt.plot(kind='bar', ax=ax, color=['#FF6B6B', '#90EE90'], width=0.7)
ax.set_title('Number of Passed and Failed Candidates by Ethnicity', fontsize=12, fontweight='bold')
ax.set_xlabel('Ethnicity')
ax.set_ylabel('Number of Candidates')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
ax.legend(['Failed', 'Passed'], loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3.7 Số thí sinh đậu, rớt theo giới tính

In [ ]:
# 3.7 Passed/Failed by gender
kqxt_by_gt = pd.crosstab(df['GT'], df['KQXT'])
print("Pass/Fail by Gender:")
print(kqxt_by_gt)

fig, ax = plt.subplots(figsize=(10, 6))
kqxt_by_gt.plot(kind='bar', ax=ax, color=['#FF6B6B', '#90EE90'], width=0.5)
ax.set_title('Number of Passed and Failed Candidates by Gender', fontsize=12, fontweight='bold')
ax.set_xlabel('Gender (M=Male, F=Female)')
ax.set_ylabel('Number of Candidates')
ax.set_xticklabels(['Female', 'Male'], rotation=0)
ax.legend(['Failed', 'Passed'], loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# PHẦN 4: TRỰC QUAN HÓA DỮ LIỆU NÂNG CAO

## 4.1 Biểu đồ đường Simple cho T1

In [ ]:
# 4.1 Simple line chart for T1
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(range(len(df)), df['T1'].values, marker='o', linestyle='-', linewidth=1.5, 
        markersize=4, color='#2E86AB', alpha=0.8)
ax.set_title('T1 (Math) Scores - Simple Line Chart', fontsize=12, fontweight='bold')
ax.set_xlabel('Student Index')
ax.set_ylabel('T1 Score')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("T1 Statistics:")
print(df['T1'].describe())

## 4.2 Tạo biến phân loại phanloait1 từ T1

In [ ]:
# 4.2 Create phanloait1 classification variable
# 0-5: kém (k), 5-7: trung bình (tb), 7-8: khá (k), 8+: giỏi (g)
def classify_t1(score):
    if score < 5:
        return 'k'  # kém (poor)
    elif score < 7:
        return 'tb'  # trung bình (average)
    elif score < 8:
        return 'k'  # khá (good) - Note: using 'k' as per instructions
    else:
        return 'g'  # giỏi (excellent)

df['phanloait1'] = df['T1'].apply(classify_t1)

print("Classification of T1:")
print(df['phanloait1'].value_counts())
print("\nFirst few rows with classification:")
print(df[['STT', 'T1', 'phanloait1']].head(20))

## 4.3 Bảng tần số cho phanloait1

In [ ]:
# 4.3 Frequency table for phanloait1
phanloait1_freq = df['phanloait1'].value_counts()
phanloait1_relative = df['phanloait1'].value_counts(normalize=True) * 100

freq_table_phanloait1 = pd.DataFrame({
    'Classification': phanloait1_freq.index,
    'Frequency': phanloait1_freq.values,
    'Relative Frequency (%)': phanloait1_relative.values
}).sort_index()

print("Frequency Table for phanloait1:")
print(freq_table_phanloait1)

## 4.4 Biểu đồ đường Multiple Line cho T1 phân loại bởi phanloait1

In [ ]:
# 4.4 Multiple line chart for T1 by phanloait1 classification
fig, ax = plt.subplots(figsize=(14, 6))

colors = {'k': '#FF6B6B', 'tb': '#4ECDC4', 'g': '#95E1D3'}
for classification in ['k', 'tb', 'g']:
    data = df[df['phanloait1'] == classification]
    ax.plot(data.index, data['T1'].values, marker='o', linestyle='-', linewidth=1.5,
            markersize=5, label=classification, color=colors.get(classification, '#999999'), alpha=0.7)

ax.set_title('T1 Scores by Classification (phanloait1) - Multiple Line Chart', 
             fontsize=12, fontweight='bold')
ax.set_xlabel('Student Index')
ax.set_ylabel('T1 Score')
ax.legend(['k (kém)', 'tb (trung bình)', 'g (giỏi)'], loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4.5 Biểu đồ Drop-line cho T1 phân loại bởi phanloait1

In [ ]:
# 4.5 Drop-line chart for T1 by phanloait1
fig, ax = plt.subplots(figsize=(14, 6))

# X-axis baseline
x_baseline = np.arange(len(df))

# Plot drop lines for each classification
for classification in ['k', 'tb', 'g']:
    data = df[df['phanloait1'] == classification]
    ax.vlines(data.index, 0, data['T1'].values, colors=colors.get(classification, '#999999'), 
              linewidth=2, alpha=0.7, label=classification)
    ax.scatter(data.index, data['T1'].values, color=colors.get(classification, '#999999'), 
               s=40, zorder=3)

ax.axhline(y=0, color='black', linewidth=1)
ax.set_title('T1 Scores by Classification (phanloait1) - Drop-line Chart', 
             fontsize=12, fontweight='bold')
ax.set_xlabel('Student Index')
ax.set_ylabel('T1 Score')
ax.legend(['k (kém)', 'tb (trung bình)', 'g (giỏi)'], loc='best')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# PHẦN 5: MỐ TẢ DỮ LIỆU VÀ KHẢO SÁT DẠNG PHÂN PHỐI

## 5.1 Mô tả và khảo sát phân phối cho T1

In [ ]:
# 5.1 Descriptive statistics and distribution analysis for T1
print("=" * 60)
print("DESCRIPTIVE STATISTICS FOR T1")
print("=" * 60)

t1_data = df['T1']
print(f"\nMeasures of Central Tendency:")
print(f"  Mean: {t1_data.mean():.4f}")
print(f"  Median: {t1_data.median():.4f}")
print(f"  Mode: {t1_data.mode().values[0]:.4f}" if len(t1_data.mode()) > 0 else "  Mode: N/A")

print(f"\nMeasures of Dispersion:")
print(f"  Variance: {t1_data.var():.4f}")
print(f"  Std Dev: {t1_data.std():.4f}")
print(f"  Range: {t1_data.max() - t1_data.min():.4f}")
print(f"  IQR (Q3-Q1): {t1_data.quantile(0.75) - t1_data.quantile(0.25):.4f}")

print(f"\nPercentiles:")
q1 = t1_data.quantile(0.25)
q2 = t1_data.quantile(0.50)
q3 = t1_data.quantile(0.75)
q_min = t1_data.quantile(0.00)
q_max = t1_data.quantile(1.00)
print(f"  Min: {q_min:.4f}")
print(f"  Q1 (25%): {q1:.4f}")
print(f"  Q2 (50%): {q2:.4f}")
print(f"  Q3 (75%): {q3:.4f}")
print(f"  Max: {q_max:.4f}")

print(f"\nSkewness and Kurtosis:")
skewness = skew(t1_data)
kurt = kurtosis(t1_data)
print(f"  Skewness: {skewness:.4f}")
print(f"  Kurtosis: {kurt:.4f}")

print(f"\nDistribution Interpretation:")
if abs(skewness) < 0.5:
    print(f"  - Distribution is approximately symmetric")
elif skewness > 0.5:
    print(f"  - Distribution is positively skewed (right-tailed)")
else:
    print(f"  - Distribution is negatively skewed (left-tailed)")
    
if kurt > 3:
    print(f"  - Distribution is leptokurtic (heavy-tailed)")
elif kurt < 3:
    print(f"  - Distribution is platykurtic (light-tailed)")
else:
    print(f"  - Distribution is mesokurtic (normal-like)")

# Normality test
stat, p_value = shapiro(t1_data)
print(f"\nShapiro-Wilk Normality Test:")
print(f"  Test Statistic: {stat:.4f}")
print(f"  P-value: {p_value:.6f}")
if p_value > 0.05:
    print(f"  Result: Data appears to be normally distributed (p > 0.05)")
else:
    print(f"  Result: Data does not appear to be normally distributed (p < 0.05)")

### 5.1.1 Box Plot cho T1

In [ ]:
# 5.1.1 Box plot for T1
fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(t1_data, vert=True, patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('lightblue')

ax.set_ylabel('T1 Score', fontsize=11)
ax.set_title('Box Plot of T1 (Math) Scores', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add labels for quartiles
q1, q2, q3 = t1_data.quantile([0.25, 0.5, 0.75])
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print("\nBox Plot Elements (10 key values):")
print(f"  1. Minimum (excluding outliers): {max(t1_data.min(), lower_fence):.4f}")
print(f"  2. Lower Fence: {lower_fence:.4f}")
print(f"  3. Q1 (First Quartile): {q1:.4f}")
print(f"  4. Q2 (Median): {q2:.4f}")
print(f"  5. Q3 (Third Quartile): {q3:.4f}")
print(f"  6. Upper Fence: {upper_fence:.4f}")
print(f"  7. Maximum (excluding outliers): {min(t1_data.max(), upper_fence):.4f}")
print(f"  8. IQR (Q3-Q1): {iqr:.4f}")
print(f"  9. Range: {t1_data.max() - t1_data.min():.4f}")
print(f"  10. Outliers count: {len(t1_data[(t1_data < lower_fence) | (t1_data > upper_fence)])}")

plt.tight_layout()
plt.show()

### 5.1.2 Histogram cho T1

In [ ]:
# 5.1.2 Histogram for T1
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(t1_data, bins=20, color='skyblue', edgecolor='navy', alpha=0.7)
ax.axvline(t1_data.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {t1_data.mean():.2f}')
ax.axvline(t1_data.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {t1_data.median():.2f}')
ax.set_title('Histogram of T1 (Math) Scores', fontsize=12, fontweight='bold')
ax.set_xlabel('T1 Score')
ax.set_ylabel('Frequency')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 5.1.3 QQ-Plot cho T1

In [ ]:
# 5.1.3 QQ-Plot for T1
from scipy.stats import probplot

fig, ax = plt.subplots(figsize=(10, 6))
probplot(t1_data, dist="norm", plot=ax)
ax.set_title('QQ-Plot of T1 (Math) Scores', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("QQ-Plot Interpretation:")
print("  - Points close to the diagonal line suggest normal distribution")
print("  - Deviations from the line indicate non-normality")

## 5.2 Phân phối T1 theo từng nhóm phanloait1

In [ ]:
# 5.2 Distribution analysis by phanloait1 groups
classifications = ['k', 'tb', 'g']
fig, axes = plt.subplots(3, 3, figsize=(16, 12))

for idx, classification in enumerate(classifications):
    data = df[df['phanloait1'] == classification]['T1']
    
    # Box plot
    axes[idx, 0].boxplot(data, vert=True)
    axes[idx, 0].set_title(f'Box Plot - {classification}', fontsize=10, fontweight='bold')
    axes[idx, 0].set_ylabel('T1 Score')
    axes[idx, 0].grid(axis='y', alpha=0.3)
    
    # Histogram
    axes[idx, 1].hist(data, bins=10, color='lightblue', edgecolor='navy', alpha=0.7)
    axes[idx, 1].set_title(f'Histogram - {classification}', fontsize=10, fontweight='bold')
    axes[idx, 1].set_ylabel('Frequency')
    axes[idx, 1].grid(axis='y', alpha=0.3)
    
    # QQ-Plot
    probplot(data, dist="norm", plot=axes[idx, 2])
    axes[idx, 2].set_title(f'QQ-Plot - {classification}', fontsize=10, fontweight='bold')
    axes[idx, 2].grid(True, alpha=0.3)
    
    # Print statistics
    print(f"\nStatistics for classification '{classification}':")
    print(f"  n: {len(data)}")
    print(f"  Mean: {data.mean():.4f}")
    print(f"  Std: {data.std():.4f}")
    print(f"  Skewness: {skew(data):.4f}")
    print(f"  Kurtosis: {kurtosis(data):.4f}")

plt.tight_layout()
plt.show()

## 5.3 Tương quan giữa DH1 và T1

In [ ]:
# 5.3 Correlation between DH1 and T1
print("=" * 60)
print("CORRELATION ANALYSIS: DH1 vs T1")
print("=" * 60)

# Covariance and Correlation
covariance = np.cov(df['DH1'], df['T1'])[0, 1]
correlation = df['DH1'].corr(df['T1'])

print(f"\nCovariance (DH1, T1): {covariance:.4f}")
print(f"Pearson Correlation: {correlation:.4f}")

# Scatter plot
fig, ax = plt.subplots(figsize=(12, 7))
ax.scatter(df['T1'], df['DH1'], alpha=0.6, s=50, color='#2E86AB')

# Add regression line
z = np.polyfit(df['T1'], df['DH1'], 1)
p = np.poly1d(z)
ax.plot(df['T1'].sort_values(), p(df['T1'].sort_values()), 
        "r--", linewidth=2, label=f'Regression line: y={z[0]:.3f}x+{z[1]:.3f}')

ax.set_title(f'Scatter Plot: DH1 vs T1 (Correlation: {correlation:.4f})', 
             fontsize=12, fontweight='bold')
ax.set_xlabel('T1 (Independent Variable)')
ax.set_ylabel('DH1 (Dependent Variable)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nInterpretation:")
if abs(correlation) < 0.3:
    print(f"  - Weak correlation between DH1 and T1")
elif abs(correlation) < 0.7:
    print(f"  - Moderate correlation between DH1 and T1")
else:
    print(f"  - Strong correlation between DH1 and T1")
    
if correlation > 0:
    print(f"  - Positive relationship: As T1 increases, DH1 tends to increase")
else:
    print(f"  - Negative relationship: As T1 increases, DH1 tends to decrease")

## 5.4 Tương quan DH1 vs T1 theo từng nhóm khu vực

In [ ]:
# 5.4 Correlation by region (KV)
print("\n" + "=" * 60)
print("CORRELATION ANALYSIS: DH1 vs T1 BY REGION")
print("=" * 60)

regions = df['KV'].unique()
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, region in enumerate(sorted(regions)):
    region_data = df[df['KV'] == region]
    corr = region_data['DH1'].corr(region_data['T1'])
    
    axes[idx].scatter(region_data['T1'], region_data['DH1'], 
                     alpha=0.6, s=50, color='#2E86AB')
    
    # Regression line
    if len(region_data) > 1:
        z = np.polyfit(region_data['T1'], region_data['DH1'], 1)
        p = np.poly1d(z)
        x_sorted = region_data['T1'].sort_values()
        axes[idx].plot(x_sorted, p(x_sorted), "r--", linewidth=2)
    
    axes[idx].set_title(f'Region {region} (n={len(region_data)}, r={corr:.4f})', 
                       fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('T1')
    axes[idx].set_ylabel('DH1')
    axes[idx].grid(True, alpha=0.3)
    
    print(f"\nRegion {region}:")
    print(f"  Count: {len(region_data)}")
    print(f"  Correlation: {corr:.4f}")

# Hide extra subplot
axes[-1].set_visible(False)

plt.tight_layout()
plt.show()

## 5.5 Tương quan giữa DH1, DH2, DH3

In [ ]:
# 5.5 Correlation matrix for DH1, DH2, DH3
print("=" * 60)
print("CORRELATION MATRIX: DH1, DH2, DH3")
print("=" * 60)

# Extract the three variables
dh_data = df[['DH1', 'DH2', 'DH3']]

# Correlation matrix
corr_matrix = dh_data.corr()
print("\nPearson Correlation Matrix:")
print(corr_matrix.round(4))

# Covariance matrix
cov_matrix = dh_data.cov()
print("\nCovariance Matrix:")
print(cov_matrix.round(4))

# Visualization of correlation matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=2, cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1, ax=ax, fmt='.3f')
ax.set_title('Correlation Matrix: DH1, DH2, DH3', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.5.1 Scatter Plot Matrix cho DH1, DH2, DH3

In [ ]:
# 5.5.1 Scatter plot matrix (pairplot)
from pandas.plotting import scatter_matrix

fig, axes = plt.subplots(3, 3, figsize=(14, 12))

# Manual scatter plot matrix
variables = ['DH1', 'DH2', 'DH3']
for i, var1 in enumerate(variables):
    for j, var2 in enumerate(variables):
        ax = axes[i, j]
        
        if i == j:
            # Diagonal: histogram
            ax.hist(df[var1], bins=15, color='lightblue', edgecolor='navy', alpha=0.7)
            ax.set_ylabel('Frequency')
        else:
            # Off-diagonal: scatter plot
            ax.scatter(df[var2], df[var1], alpha=0.5, s=30, color='#2E86AB')
            
            # Add regression line
            z = np.polyfit(df[var2], df[var1], 1)
            p = np.poly1d(z)
            x_sorted = df[var2].sort_values()
            ax.plot(x_sorted, p(x_sorted), "r--", linewidth=1.5, alpha=0.7)
        
        ax.set_xlabel(var2 if i == 2 else '')
        ax.set_ylabel(var1 if j == 0 else '')
        ax.grid(True, alpha=0.3)

plt.suptitle('Scatter Plot Matrix: DH1, DH2, DH3', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("\nInterpretations:")
print(f"  DH1-DH2 Correlation: {df['DH1'].corr(df['DH2']):.4f}")
print(f"  DH1-DH3 Correlation: {df['DH1'].corr(df['DH3']):.4f}")
print(f"  DH2-DH3 Correlation: {df['DH2'].corr(df['DH3']):.4f}")

## KÊTLUẬN

### Tóm tắt các phân tích chính:

1. **Thống kê Dữ liệu**: 
   - Đã sắp xếp dữ liệu DH1 và DH2 theo các tiêu chí khác nhau
   - Tạo pivot table thống kê theo KT, KV, DT

2. **Trình bày Dữ liệu**:
   - Phân tích tần số và tần suất cho các biến GT, US_TBM1, US_TBM2, US_TBM3
   - Lọc dữ liệu theo các điều kiện cụ thể

3. **Trực quan hóa Nhóm Phân loại**:
   - Biểu đồ cột cho các nhóm phân loại
   - Phân tích KQXT theo khối thi, khu vực, dân tộc, giới tính

4. **Trực quan Nâng cao**:
   - Biểu đồ đường Simple cho T1
   - Tạo biến phân loại phanloait1
   - Biểu đồ Multiple Line và Drop-line

5. **Phân tích Phân phối**:
   - Box Plot, Histogram, QQ-Plot cho T1
   - Phân tích phân phối theo nhóm phanloait1
   - Kiểm định chuẩn Shapiro-Wilk

6. **Phân tích Tương quan**:
   - Tương quan DH1-T1 tổng thể và theo khu vực
   - Ma trận tương quan DH1, DH2, DH3
   - Scatter plot matrix

**Hoàn thành**: Tất cả các phần từ 1-5 của bài tập đã được thực hiện chi tiết với các biểu đồ, bảng thống kê và phân tích.